# Modelagem e resultados

Continuação do notebook 01. Aqui ficam o ajuste de hiperparâmetros, a comparação dos cinco modelos, o teste final e a importância das variáveis (seções 3 e 4 do artigo).

As funções estão em `analise/modelagem.py` e `analise/figuras_resultados.py`. Os resultados são gravados em `dados/resultados/`, e a tabela e as figuras do artigo em `saidas/tabelas/` e `saidas/figuras/`. O script `analise/experimento_final.py` faz as mesmas etapas sem o notebook.

Leva uns 15 minutos, quase tudo na busca de hiperparâmetros. As sementes são fixas, então o resultado se repete com as mesmas versões das bibliotecas.

## Configuração

No Colab, envie o arquivo `notebooks/projeto_colab.zip` quando a célula pedir.

In [ ]:
import sys
import zipfile
from pathlib import Path

# acha a pasta do projeto (no Colab ela vem do projeto_colab.zip)
def achar_projeto():
    for pasta in [Path.cwd(), Path.cwd().parent, Path("/content/projeto")]:
        if (pasta / "dados" / "dataset_modelagem.csv").exists():
            return pasta

RAIZ = achar_projeto()
if RAIZ is None:
    from google.colab import files
    enviado = files.upload()  # escolha o arquivo projeto_colab.zip
    with zipfile.ZipFile(next(iter(enviado))) as z:
        z.extractall("/content/projeto")
    RAIZ = achar_projeto()
    %pip install -q shap

sys.path.insert(0, str(RAIZ / "analise"))
sys.path.insert(0, str(RAIZ / "scraper"))
print("projeto em", RAIZ)

In [ ]:
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import lightgbm
import numpy as np
import pandas as pd
import shap
import sklearn
import xgboost
from IPython.display import Image, display
from sklearn.model_selection import GroupKFold

import modelagem as m
import figuras_resultados as fr

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)

# mostra a figura salva
def mostrar(nome):
    display(Image(filename=str(fr.e.FIG / f"{nome}.png"), width=560))

SAIDA = m.RESULTADOS
SAIDA.mkdir(parents=True, exist_ok=True)

# as versões influenciam os números; as do artigo são scikit-learn 1.9.1, xgboost 3.4.1, lightgbm 4.7.0 e shap 0.52.0
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__,
      "| lightgbm", lightgbm.__version__, "| shap", shap.__version__)

## Dados

Só os anúncios com preço firme. Os "A partir de" são preço de tabela de lançamento, então ficam de fora e só voltam na análise de sensibilidade.

A divisão 70/30 é embaralhada porque o arquivo vem ordenado do mais caro para o mais barato. O teste só é usado no final.

In [ ]:
base = m.carregar_base(incluir_lancamentos=False)
base_completa = m.carregar_base(incluir_lancamentos=True)

# 70% treino, 30% teste (o teste fica guardado para o final)
X_treino, X_teste, y_treino, y_teste = m.dividir(base)

print("base completa:", len(base_completa), "| 'A partir de':", int(base_completa["preco_lancamento"].sum()))
print("preço firme:", len(base), "| treino:", len(X_treino), "| teste:", len(X_teste))
print(len(m.NUMERICAS), "numéricas e", len(m.CATEGORICAS), "categóricas")
pd.DataFrame({"treino": y_treino.describe(), "teste": y_teste.describe()}).map("{:,.0f}".format)

## Busca de hiperparâmetros

Para cada modelo de árvore são sorteadas 50 combinações das faixas da tabela de hiperparâmetros do artigo, e cada uma é testada com validação cruzada de 5 partições no treino. Fica a de menor MAPE. A Regressão Linear não tem o que ajustar.

Todos os modelos usam o mesmo pipeline: nas numéricas, mediana nos vazios (com uma coluna marcando o vazio) e escala de 0 a 1; nas categóricas, one-hot, com os bairros de menos de 10 anúncios juntos numa categoria só. O alvo é o log do preço, e as previsões voltam para reais antes das métricas.

In [ ]:
padrao = m.modelos_referencia()
ajustados, historico = m.ajustar_hiperparametros(padrao, X_treino, y_treino)
historico.to_csv(SAIDA / "busca_hiperparametros.csv", index=False, encoding="utf-8-sig")

melhores = m.melhores_hiperparametros(ajustados)
melhores.to_csv(SAIDA / "melhores_hiperparametros.csv", index=False, encoding="utf-8-sig")

# MAPE das 50 combinações de cada modelo
historico.groupby("modelo")["MAPE"].agg(["min", "median", "max"]).rename(
    columns={"min": "melhor", "median": "mediana", "max": "pior"})

In [ ]:
# configuração escolhida e se ela ficou na ponta da faixa testada
def limites(espaco):
    if hasattr(espaco, "support"):   # distribuições do scipy
        return espaco.support()
    numeros = [v for v in espaco if isinstance(v, (int, float)) and not isinstance(v, bool)]
    return (min(numeros), max(numeros)) if numeros else (None, None)

linhas = []
for _, r in melhores.iterrows():
    lo, hi = limites(m.ESPACOS_BUSCA[r.modelo][m._P + r.hiperparametro])
    no_limite = isinstance(r.valor, (int, float)) and lo is not None and (np.isclose(r.valor, lo) or np.isclose(r.valor, hi))
    linhas.append({**r, "faixa": f"{lo:g} a {hi:g}" if lo is not None else "", "no limite": "sim" if no_limite else ""})
pd.DataFrame(linhas).set_index(["modelo", "hiperparametro"])

## Validação cruzada repetida (5 x 5)

Validação cruzada de 5 partições repetida 5 vezes no treino, com outra semente, o que dá 25 resultados por modelo. Roda com os hiperparâmetros padrão e com os ajustados, para ver quanto o ajuste ajudou.

In [ ]:
cv_padrao = m.avaliar_cv(padrao, X_treino, y_treino, cv=m.cv_repetida())
cv_ajust = m.avaliar_cv(ajustados, X_treino, y_treino, cv=m.cv_repetida())
pd.concat([cv_padrao.assign(configuracao="padrao"), cv_ajust.assign(configuracao="ajustado")]) \
    .to_csv(SAIDA / "cv_repetida_por_particao.csv", index=False, encoding="utf-8-sig")

def resumo_cv(por_particao, rotulo):
    r = m.resumir(por_particao)
    r.columns = [f"{a}_{b}" for a, b in r.columns]
    return r.assign(configuracao=rotulo)

resumo = pd.concat([resumo_cv(cv_padrao, "padrao"), resumo_cv(cv_ajust, "ajustado")])
resumo.to_csv(SAIDA / "cv_repetida_resumo.csv", encoding="utf-8-sig")

# média das 25 partições, com os hiperparâmetros ajustados
aj = resumo[resumo.configuracao == "ajustado"]
pd.DataFrame({
    "MAPE (%)": aj.MAPE_mean.map("{:.2f}".format) + " ± " + aj.MAPE_std.map("{:.2f}".format),
    "R²": aj.R2_mean.map("{:.3f}".format),
    "RMSE (R$)": aj.RMSE_mean.map("{:,.0f}".format),
    "MAE (R$)": aj.MAE_mean.map("{:,.0f}".format),
    "treino por partição (s)": aj.tempo_s_mean.map("{:.2f}".format),
})

In [ ]:
# quanto o ajuste baixou o MAPE
pa = resumo[resumo.configuracao == "padrao"]["MAPE_mean"]
pd.DataFrame({"MAPE padrão": pa, "MAPE ajustado": aj["MAPE_mean"],
              "redução (p.p.)": pa - aj["MAPE_mean"]}).sort_values("MAPE ajustado")

## Comparação estatística

As 25 partições usam quase os mesmos dados de treino, então não são independentes, e o teste t comum acharia diferenças que não existem. Por isso foi usado o teste t corrigido de Nadeau e Bengio (2003): a variância da diferença é multiplicada por 1/k + n_validação/n_treino (aqui, 1/25 + 1/4), em vez de só dividida por k.

Com 5 modelos são 10 pares, e os valores-p são ajustados pelo método de Holm, para que a chance de ao menos um falso positivo continue em 5%. O posto médio só resume a ordem dos modelos.

In [ ]:
comp = m.comparar_estatisticamente(cv_ajust)
comp["pares"].to_csv(SAIDA / "comparacao_pares.csv", index=False, encoding="utf-8-sig")
print("posto médio no MAPE (1 = melhor):", comp["postos_medios"].round(2).to_dict())

# a_melhor_em: em quantas das 25 partições o modelo A teve MAPE menor que o B
comp["pares"][["modelo_a", "modelo_b", "diferenca_media", "p", "p_holm", "significativo", "a_melhor_em"]]

## Validação agrupada por prédio

Várias unidades do mesmo prédio são quase iguais. Na divisão aleatória, uma pode cair no treino e a outra na validação, o que facilita. Aqui todas as unidades de um prédio (mesma coordenada) ficam na mesma partição.

In [ ]:
grupos = m.grupos_por_predio(base, X_treino.index)
cv_predio = m.avaliar_cv(ajustados, X_treino, y_treino, cv=GroupKFold(m.N_PARTICOES), grupos=grupos)
cv_predio.to_csv(SAIDA / "cv_agrupada_predio.csv", index=False, encoding="utf-8-sig")
print(len(X_treino), "anúncios de treino em", len(set(grupos)), "prédios")

agr = m.resumir(cv_predio)[("MAPE", "mean")]
pd.DataFrame({"MAPE aleatória": aj["MAPE_mean"], "MAPE por prédio": agr,
              "aumento (p.p.)": agr - aj["MAPE_mean"]}).sort_values("MAPE por prédio")

## Sensibilidade aos anúncios "A partir de"

Os modelos ajustados são treinados de novo com os lançamentos incluídos só no treino, com uma coluna marcando quais são, e testados no mesmo conjunto de teste (só preço firme).

In [ ]:
sens, n_lanc = m.sensibilidade_lancamentos(ajustados, base, base_completa)
sens.to_csv(SAIDA / "sensibilidade_lancamentos.csv", index=False, encoding="utf-8-sig")
print(n_lanc, "lançamentos adicionados ao treino")

t_sens = sens.pivot(index="modelo", columns="treino", values="MAPE")[["sem lançamentos", "com lançamentos"]]
t_sens.assign(**{"variação (p.p.)": t_sens["com lançamentos"] - t_sens["sem lançamentos"]})

## Teste final

Cada modelo ajustado é treinado com todo o treino e testado uma vez no conjunto de teste. Os modelos treinados ficam salvos em `dados/resultados/modelos/`.

In [ ]:
teste, treinados = m.avaliar_teste(ajustados, X_treino, y_treino, X_teste, y_teste)
teste.to_csv(SAIDA / "teste_metricas.csv", encoding="utf-8-sig")
m.salvar_modelos(treinados, SAIDA / "modelos")
teste.assign(MAE=teste.MAE.map("{:,.0f}".format), RMSE=teste.RMSE.map("{:,.0f}".format))

In [ ]:
# tabela de resultados e figura do MAPE nas 25 partições, usadas no artigo
fr.tabela_resultados(resumo.reset_index(), teste)
fr.fig_mape_particoes(pd.concat([cv_padrao.assign(configuracao="padrao"), cv_ajust.assign(configuracao="ajustado")]), teste)
print((fr.e.TAB / "resultados_modelos.tex").read_text(encoding="utf-8"))
mostrar("mape_particoes")

### Onde estão os maiores erros

Os modelos de árvore ganham mais no MAPE do que no RMSE. Como o RMSE pesa mais os erros grandes, vale ver onde eles estão no melhor modelo.

In [ ]:
melhor = teste.index[0]
prev = pd.Series(treinados[melhor].predict(X_teste), index=X_teste.index)
erro2 = (y_teste - prev) ** 2
q4 = y_teste >= y_teste.quantile(0.75)
dez = erro2.nlargest(10).index

print("modelo:", melhor)
print(f"quarto mais caro do teste: {erro2[q4].sum() / erro2.sum():.0%} do erro quadrático")
print(f"dez maiores erros: {erro2[dez].sum() / erro2.sum():.0%} do erro quadrático,",
      (y_teste[dez] > 1_200_000).sum(), "deles acima de R$ 1,2 milhão")
print("anúncios acima de R$ 2 milhões no treino:", (y_treino > 2_000_000).sum())
print(f"preço mediano da base: R$ {base_completa['preco'].median():,.0f}")
print(f"anúncios no Centro: {base_completa['bairro'].eq('Centro').mean():.0%}")

# os dez maiores erros
pd.DataFrame({"preço real": y_teste[dez], "previsto": prev[dez].round(0),
              "erro (%)": ((prev[dez] - y_teste[dez]) / y_teste[dez] * 100).round(1)}).map("{:,.1f}".format)

## Importância por permutação

No conjunto de teste, cada variável é embaralhada 20 vezes e mede-se quanto o MAPE aumenta, em pontos percentuais. Funciona com as variáveis originais (o bairro conta como uma só) e serve para os cinco modelos, inclusive a Regressão Linear.

In [ ]:
perm = m.importancia_permutacao(treinados, X_teste, y_teste)
perm.to_csv(SAIDA / "importancia_permutacao.csv", index=False, encoding="utf-8-sig")

tab_perm = perm.pivot(index="variavel", columns="modelo", values="aumento_MAPE_pp")[fr.ORDEM]
print("mais importante em cada modelo:", tab_perm.idxmax().to_dict())
tab_perm.sort_values("Gradient Boosting", ascending=False).head(12)

Variáveis muito correlacionadas, como área, dormitórios, suítes e vagas, dividem a importância entre si e a ordem muda de um modelo para outro. Por isso a importância também é medida por grupos, embaralhando todas as variáveis do grupo juntas.

In [ ]:
for grupo, cols in fr.GRUPOS.items():
    print(grupo + ":", ", ".join(cols))

imp = fr.importancia_grupos(treinados, X_teste, y_teste)
imp.to_csv(SAIDA / "importancia_grupos.csv", index=False, encoding="utf-8-sig")
fr.fig_importancia_grupos(imp)
mostrar("importancia_grupos")
imp.pivot(index="grupo", columns="modelo", values="aumento_MAPE_pp")[fr.ORDEM] \
    .sort_values("Gradient Boosting", ascending=False)

## SHAP

Feito no melhor modelo de árvore do teste. Para cada imóvel, o SHAP parte da previsão média e dá a cada variável uma contribuição, positiva ou negativa; somando tudo com a média, chega na previsão do imóvel. Como o alvo é o log do preço, uma contribuição de 0,15 equivale a uns 16% no preço (e^0,15 - 1).

A figura mostra as colunas do modelo (a falta do número de vagas, por exemplo, aparece separada). O resumo por variável soma as colunas de um mesmo bairro.

In [ ]:
melhor_arvore = next(n for n in teste.index if n in m.ESPACOS_BUSCA)
_, _, shap_resumo = m.valores_shap(treinados[melhor_arvore], X_teste)
shap_resumo.rename("media_abs_shap").to_csv(SAIDA / "shap_resumo.csv", encoding="utf-8-sig")

shap_dir = fr.fig_shap(treinados[melhor_arvore], X_teste)
shap_dir.to_csv(SAIDA / "shap_direcao.csv", encoding="utf-8-sig")
print("modelo:", melhor_arvore)
mostrar("shap_gradient_boosting")

# direcao_spearman positiva: quanto maior a variável, maior o preço previsto
shap_dir.head(15).assign(**{"efeito médio no preço (%)": (np.exp(shap_dir.head(15)["media_abs_shap"]) - 1) * 100})

In [ ]:
# variáveis de localização comparadas com a área
localizacao = ["bairro", "latitude", "longitude", "dist_shopping_km", "dist_hospital_km", "dist_ensino_km"]
pd.DataFrame({"média |SHAP|": shap_resumo[localizacao],
              "fração da área": shap_resumo[localizacao] / shap_resumo["area_util_m2"]})

## Resumo

In [ ]:
resumo_json = {
    "n_base": len(base), "n_treino": len(X_treino), "n_teste": len(X_teste),
    "n_lancamentos_sensibilidade": n_lanc,
    "busca": {"n_iteracoes": m.N_ITERACOES_BUSCA, "particoes": m.N_PARTICOES, "criterio": m.METRICA_SELECAO},
    "cv_repetida": {"particoes": m.N_PARTICOES, "repeticoes": m.N_REPETICOES},
    "postos_medios": comp["postos_medios"].to_dict(),
    "pares_significativos_holm": comp["pares"].loc[comp["pares"]["significativo"], ["modelo_a", "modelo_b"]].values.tolist(),
    "melhor_modelo_teste": teste.index[0], "melhor_arvore_shap": melhor_arvore,
}
with open(SAIDA / "experimento_resumo.json", "w", encoding="utf-8") as f:
    json.dump(resumo_json, f, ensure_ascii=False, indent=2)
print(json.dumps(resumo_json, ensure_ascii=False, indent=2))

## Conclusão

- Os três modelos de boosting ficaram empatados, com MAPE perto de 15,5% na validação cruzada e 15% a 15,4% no teste.
- Eles são melhores que a Regressão Linear com significância estatística (ganharam nas 25 partições). As outras diferenças não passaram no teste com a correção de Holm.
- Os maiores erros estão nos imóveis caros, que são poucos na base.
- Os resultados quase não mudam agrupando por prédio nem incluindo os lançamentos.
- O que mais pesa no preço é o tamanho e o padrão do imóvel. A localização ajuda pouco depois disso.